In [1]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("EMR-Local-Dev")
         .getOrCreate())

df = spark.read.parquet('s3://training-data-23421/transformed-output/71e269cf-15be-497f-a7f1-01c0023856f7/customer_data_processed.parquet/part-00000-a07efa28-9d7c-4e31-9c93-0679973bf3d1-c000.snappy.parquet')
df.show(5)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 01:33:48 WARN EC2MetadataUtils: Unable to retrieve the requested metadata (/latest/dynamic/instance-identity/document). Failed to connect to service endpoint: 
com.amazon.ws.emr.hadoop.fs.shaded.com.amazonaws.SdkClientException: Failed to connect to service endpoint: 
	at com.amazon.ws.emr.hadoop.fs.shaded.com.amazonaws.internal.EC2ResourceFetcher.doReadResource(EC2ResourceFetcher.java:112) ~[emrfs-hadoop-assembly-2.60.0.jar:?]
	at com.amazon.ws.emr.hadoop.fs.shaded.com.amazonaws.internal.EC2ResourceFetcher.doReadResource(EC2ResourceFetcher.java:70) ~[emrfs-hadoop-assembly-2.60.0.jar:?]
	at com.amazon.ws.emr.hadoop.fs.shaded.com.amazonaws.internal.InstanceMetadataServiceResourceFetcher.readResource(InstanceMetadataServiceResourceFetcher.java:75) ~[emrfs-hadoop-assembly-2.60.0.jar:?]
	at com.amazon.ws.emr.hadoop.fs.shaded.com.amazonaws.internal.EC2Re

+-----------+---+-------------+--------------+---------------+------------+------+--------------------+-----------+------------+-----------+-----------+----------------------+---------------------+----------------------+----------------------------+-------------------------+--------------------------+----------------------------+----------------------------+---------------------------+---------------------------+----------------------------+---------------------------+---------------------------+----------------------------+----------------------------+----------------------------+---------------------------+---------------------------+----------------------------+----------------------------+---------------------------+---------------------------+----------------------------+---------------------------+----------------------------+
|Customer_ID|Age|Annual_Income|Spending_score|Num_of_Children|Credit_Score|Target|        created_time|Gender_Male|Region_North|Region_East|Region_West|Mar

In [2]:
import time
from pyspark.sql import functions as F
from datetime import datetime, timezone

now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ")

utc_now = datetime.now(timezone.utc)
utc_timestamp = utc_now.timestamp()

df_prepared = df.withColumn("api_invocation_time", F.lit(int(utc_timestamp * 1000))) \
                .withColumn("write_time", F.lit(int(utc_timestamp * 1000))) \
                .withColumn("is_deleted", F.lit(False))
df_prepared.select('api_invocation_time', 'write_time', 'is_deleted', 'created_time').show(5, False)

+-------------------+-------------+----------+------------------------+
|api_invocation_time|write_time   |is_deleted|created_time            |
+-------------------+-------------+----------+------------------------+
|1774489040413      |1774489040413|false     |2026-03-25T17:42:00.394Z|
|1774489040413      |1774489040413|false     |2026-03-25T17:42:00.394Z|
|1774489040413      |1774489040413|false     |2026-03-25T17:42:00.394Z|
|1774489040413      |1774489040413|false     |2026-03-25T17:42:00.394Z|
|1774489040413      |1774489040413|false     |2026-03-25T17:42:00.394Z|
+-------------------+-------------+----------+------------------------+
only showing top 5 rows



In [3]:
df_prepared = df_prepared.withColumn("year", F.year("created_time")) \
                .withColumn("month", F.lpad(F.month("created_time"), 2, '0')) \
                .withColumn("day", F.lpad(F.dayofmonth("created_time"), 2, '0')) \
                .withColumn("hour", F.lpad(F.hour("created_time"), 2, '0'))
df_prepared.select('year', 'month', 'day', 'hour', 'api_invocation_time', 'write_time', 'is_deleted', 'created_time').show(5, False)

+----+-----+---+----+-------------------+-------------+----------+------------------------+
|year|month|day|hour|api_invocation_time|write_time   |is_deleted|created_time            |
+----+-----+---+----+-------------------+-------------+----------+------------------------+
|2026|03   |25 |17  |1774489040413      |1774489040413|false     |2026-03-25T17:42:00.394Z|
|2026|03   |25 |17  |1774489040413      |1774489040413|false     |2026-03-25T17:42:00.394Z|
|2026|03   |25 |17  |1774489040413      |1774489040413|false     |2026-03-25T17:42:00.394Z|
|2026|03   |25 |17  |1774489040413      |1774489040413|false     |2026-03-25T17:42:00.394Z|
|2026|03   |25 |17  |1774489040413      |1774489040413|false     |2026-03-25T17:42:00.394Z|
+----+-----+---+----+-------------------+-------------+----------+------------------------+
only showing top 5 rows



In [4]:
s3_uri = "s3://feature-store-786q23/111/sagemaker/us-east-2/offline-store/user-behavior-v1-1774458441/data/"

df_prepared.write \
    .partitionBy("year", "month", "day", "hour") \
    .mode("append") \
    .parquet(s3_uri)

In [4]:
##%%
# Load transformed DF into feature store
feature_store_manager = FeatureStoreManager()
feature_definitions = feature_store_manager.load_feature_definitions_from_schema(df)

feature_group_arn = "arn:aws:sagemaker:us-east-2:111:feature-group/user-behavior-v1"

# Ingest by default. The connector will leverage PutRecord API to ingest your data in stream
# https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_feature_store_PutRecord.html
feature_store_manager.ingest_data(input_data_frame=df, feature_group_arn=feature_group_arn, target_stores=["OfflineStore"])

# To select the target stores for ingestion, you can specify the target store as the paramter
# If OnlineStore is selected, the connector will leverage PutRecord API to ingest your data in stream
# feature_store_manager.ingest_data(input_data_frame=encoded_df, feature_group_arn=feature_group_arn, target_stores=["OfflineStore", "OnlineStore"])

# If only OfflineStore is selected, the connector will batch write the data to offline store directly
# feature_store_manager.ingest_data(input_data_frame=encoded_df, feature_group_arn=feature_group_arn, target_stores=["OfflineStore"])

# To retrieve the records failed to be ingested by spark connector
failed_records_df = feature_store_manager.get_failed_stream_ingestion_data_frame()

failed_records_df

26/03/25 19:14:46 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/03/25 19:14:46 WARN CredentialProviderListFactory: Credentials option fs.s3a.aws.credentials.provider contains AWS v1 SDK entry com.amazonaws.auth.ContainerCredentialsProvider
26/03/25 19:14:46 WARN S3AFileSystem: Getting region for bucket feature-store-786q23 from S3, this will slow down FS initialisation. To avoid this, set the region using property fs.s3a.bucket.feature-store-786q23.endpoint.region
